In [1]:
#conda activate burnseverity
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

import requests
import json

import geopandas as gpd

import rasterio as rio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.mask import mask
from rasterio import features
from rasterio.plot import show_hist

from shapely.geometry import shape, mapping
from shapely.ops import unary_union

import validation as val


### Validation goals

#### Questions
1. Difference between our dNBR and dNBR from BAIR?
2. Are the two different indices (dNBR and RBR) truly different in our tool?
3. Do our boundaries and those from CalFire match?
4. Sensitivity around chosen time windows?

#### Metrics
1. Recoarse Sentinel to Landsat, $R^2$
2. Compare $var(dNBR)$ vs $var(RBR)$
3. Percent overlap
4. Before period fixed through three weeks. After ${5, 10, 15, 21, 30, 45, 60, 90}$. Criteria: stable and low variance

In [2]:
fires = pd.read_csv('fire_processing_jobs.csv')
calfire = gpd.read_file('Validation_Fire_Perimeters_2015_2024.shp')

In [ ]:
result = val.process_fire_metrics('GEOLOGY', 5, 'alarm', fires, calfire)


  dNBR URL: 3e2b-47ee-9d10-539c609af107/fire_severity/dnbr.tif
  RdNBR URL: e2b-47ee-9d10-539c609af107/fire_severity/rdnbr.tif
    dnbr: min=-0.1052, max=0.3757, sum=2168.5247, n_positive=10286
    rdnbr: min=-1.9053, max=30971466.0000, sum=59288040.0000, n_positive=10286


In [ ]:
# Process all fires, date modes, and date ranges
results_list = []
current_key = None

for _, row in fires.iterrows():
    key = (row['fire_name'], row['date_mode'])
    if key != current_key:
        print(f"Processing {row['fire_name']} ({row['date_mode']})...")
        current_key = key

    result = val.process_fire_metrics(row['fire_name'], row['post_fire_days'], row['date_mode'], fires, calfire)

    if result is False:
        continue

    # result is now a list of dicts (one per metric)
    results_list.extend(result)

# Create DataFrame
metrics_df = pd.DataFrame(results_list)

# Split by metric and save separate geopackages
dnbr_df = metrics_df[metrics_df['metric'] == 'dnbr'].copy()
rbr_df = metrics_df[metrics_df['metric'] == 'rbr'].copy()

# Save dNBR as GeoPackage
dnbr_gdf = gpd.GeoDataFrame(
    dnbr_df,
    geometry='filtered_polygon',
    crs='EPSG:4326'
)
dnbr_gdf.to_file('validation_metrics_dnbr.gpkg', driver='GPKG')

# Save RBR as GeoPackage
rbr_gdf = gpd.GeoDataFrame(
    rbr_df,
    geometry='filtered_polygon',
    crs='EPSG:4326'
)
rbr_gdf.to_file('validation_metrics_rbr.gpkg', driver='GPKG')

# Also save combined attributes as CSV for quick viewing
metrics_df.drop('filtered_polygon', axis=1).to_csv('validation_metrics.csv', index=False)

print(f"\nFinal table shape: {metrics_df.shape}")
print(f"dNBR rows: {len(dnbr_df)}, RBR rows: {len(rbr_df)}")
print("Saved spatial data to validation_metrics_dnbr.gpkg and validation_metrics_rbr.gpkg")
print("Saved CSV to validation_metrics.csv")

metrics_df.head()


Processing COFFEE POT...
Processing SENTINEL...
Processing SIMPSON...
Processing YORK...
  Job YORK_2023-07-28_5 is pending
  Job YORK_2023-07-28_10 is pending
  Job YORK_2023-07-28_15 is pending
  Job YORK_2023-07-28_21 is pending
  Job YORK_2023-07-28_30 is pending
  Job YORK_2023-07-28_45 is pending
  Job YORK_2023-07-28_60 is pending
  Job YORK_2023-07-28_90 is pending
Processing REDWOOD...
Processing GEOLOGY...
Processing VALLEY...
Processing SYCAMORE...
Processing SUMMIT...
Processing ELK TRAIL...
Processing AVALANCHE...
Processing KNP Complex...
  Job KNP Complex_2021-09-10_5 is pending
  Job KNP Complex_2021-09-10_10 is pending
  Job KNP Complex_2021-09-10_15 is pending
  Job KNP Complex_2021-09-10_21 is pending
  Job KNP Complex_2021-09-10_30 is pending
  Job nan is pending
  Job nan is pending
  Job nan is pending
Processing MOJAVE...
  Job nan is pending
Processing POND...
Processing LOST...
Processing HART...
Processing CASTLE...
Processing DOME...
  Job DOME_2020-08-15_5 i

,fire_name,fire_days,fire_event_name,metric,calfire_mean,calfire_var,calfire_n_removed,calfire_pct_removed,filtered_mean,filtered_var,filtered_n_removed,filtered_pct_removed,filtered_polygon
0,COFFEE POT,5,COFFEE POT_2024-08-03_5,dnbr,0.057040,0.005023,129,0.073649,0.075085,0.004029,129,0.088878,MULTIPOLYGON (((-118.78054641229755 36.3526655...
1,COFFEE POT,5,COFFEE POT_2024-08-03_5,rbr,0.030831,0.001339,0,0.000000,0.040974,0.000960,0,0.000000,MULTIPOLYGON (((-118.78054641229755 36.3526655...
2,COFFEE POT,10,COFFEE POT_2024-08-03_10,dnbr,0.054352,0.004792,108,0.061660,0.073523,0.003465,105,0.072729,MULTIPOLYGON (((-118.7948966008226 36.35266556...
3,COFFEE POT,10,COFFEE POT_2024-08-03_10,rbr,0.030573,0.001393,0,0.000000,0.041393,0.000941,0,0.000000,MULTIPOLYGON (((-118.7948966008226 36.35266556...
4,COFFEE POT,15,COFFEE POT_2024-08-03_15,dnbr,0.027723,0.005877,95,0.054238,0.063466,0.003403,80,0.066843,MULTIPOLYGON (((-118.79507597817916 36.3524830...
